In [ ]:
"""
STEP 1 - AUDIT THE PhoNIX DATABASE
================================================================================

    python scripts/01_audit_phonix.py

WHAT THIS SCRIPT IS FOR
-----------------------
Before we model anything, we check that the file on disk is what the paper says
it is. That is the whole job of this script. It trains nothing and predicts
nothing.

This sounds boring, and it is the single most valuable hour in the project.
Every later number - every accuracy, every candidate - inherits whatever is
wrong here. If the file is a different release than the paper describes, or the
two heat channels do not add up, we want to find out now, not in week six.

The paper is papers/phonix_npjcm_2026.pdf. It claims:

    6,641 unique materials, 7,342 data points, 701 duplicated material entries,
    all properties evaluated at 300 K.

We are going to check those claims against data/phonix/data_all.csv ourselves.

THE SCIENCE, IN ONE PARAGRAPH
-----------------------------
Heat travels through a crystal by two mechanisms, and PhoNIX reports them
separately:

    kp   "Peierls" conductivity. The textbook picture: phonons (packets of
         atomic vibration) behave like particles bouncing around.
    kc   "coherence" conductivity. In complex, messy crystals the particle
         picture breaks down and heat tunnels between vibration modes instead -
         wave-like rather than particle-like transport.
    klat total lattice thermal conductivity = kp + kc.

Most crystals are dominated by kp. A minority are dominated by kc, and those
tend to conduct heat extremely badly - which is exactly what you want for a
thermoelectric or a thermal-barrier coating. Finding those is the project.

We define the coherence fraction:

    f_c = kc / (kp + kc)

and call a material "coherence-dominated" when f_c >= 0.5, meaning more than
half its heat moves by the wave-like channel.

WHAT IT WRITES
--------------
    results/01_audit_summary.csv     every check, its result, and pass/fail
    results/01_target_distribution.png   what the numbers we are predicting look like
"""

# ============================================================================
# IMPORTS - bringing in code other people wrote
# ============================================================================
# In Python, `import X` loads a library so you can use it. `import X as Y` gives
# it a shorter nickname. These four nicknames (pd, np, plt, Path) are near
# universal conventions - you will see them in almost every script online.

import pandas as pd              # pandas: spreadsheets in Python. "pd" by convention.
import numpy as np               # numpy: fast math on arrays of numbers.
import matplotlib                # matplotlib: plotting.
matplotlib.use("Agg")            # "Agg" = draw to a file, never try to open a window.
                                 # Without this a script can hang on a machine
                                 # with no display, waiting for a window nobody sees.
import matplotlib.pyplot as plt  # the actual drawing commands live in .pyplot
from pathlib import Path         # Path: a tidy way to handle file locations.
                                 # `from X import Y` takes ONE thing out of a
                                 # library instead of the whole thing.


# ============================================================================
# WHERE THINGS LIVE
# ============================================================================
# A variable in CAPITALS is a convention meaning "this is a constant - set once
# at the top, never changed while the script runs." Python does not enforce it;
# it is a message to whoever reads the code.

# __file__ is a built-in variable holding THIS script's own location.
# .resolve() turns it into a full path. .parents[1] goes up one directory:
# this file is in scripts/, so parents[1] is the project folder itself.
# Doing it this way means the script works no matter which folder you run it
# from - a hard-coded "/Users/mac/Desktop/..." would break the moment anything moved.
ROOT = Path(__file__).resolve().parents[1]

# The "/" here does NOT mean division. pathlib redefines it to mean "join these
# into a path", so this reads almost like the path itself.
DATA = ROOT / "data" / "phonix" / "data_all.csv"
RESULTS = ROOT / "results"

# What the paper claims. Written out explicitly so the comparison below is a
# real check against a fixed reference, not a number we invent after looking.
PAPER_N_MATERIALS = 6641
PAPER_N_POINTS = 7342
PAPER_N_DUPLICATES = 701

# The threshold that defines our positive class. Pulled out into a named
# constant so that if we ever change it, we change it in exactly one place.
DOMINANCE_THRESHOLD = 0.5


# ============================================================================
# A SMALL HELPER FOR COLLECTING RESULTS
# ============================================================================
# A "function" is a named, reusable block of code. `def` starts one. The names
# in the brackets are its inputs.
#
# `checks` below is a LIST - an ordered collection, written with [] - and each
# thing we put in it is a DICTIONARY, written with {}. A dictionary stores
# labelled values: {"name": "rows", "value": 7341} lets you later ask for
# entry["value"] by name rather than remembering a position number.

checks = []   # starts empty; the function below fills it up


def record(name, found, expected=None, note=""):
    """Store one check so we can print and save them all together at the end.

    The `=None` and `=""` are DEFAULT VALUES: if the caller does not supply
    `expected` or `note`, Python fills in those defaults. That is why some calls
    below pass two arguments and others pass four.
    """
    # `if expected is None:` - we did not give this check something to compare
    # against, so it is informational rather than pass/fail.
    if expected is None:
        status = "info"
    elif found == expected:
        status = "PASS"
    else:
        status = "MISMATCH"

    # .append() adds one item to the end of a list.
    checks.append({"check": name, "found": found, "expected": expected,
                   "status": status, "note": note})

    # An f-string (the f before the quote) lets you drop variables straight into
    # text inside {curly braces}. The `:<46` part means "pad this to 46
    # characters, left-aligned", which is what keeps the output in neat columns.
    exp = "" if expected is None else f"  (paper: {expected})"
    print(f"  {name:<46} {str(found):>14}  {status:<9}{exp}")


# ============================================================================
# THE MAIN WORK
# ============================================================================
def main():
    print("=" * 78)
    print("STEP 1 - AUDITING PhoNIX")
    print("=" * 78)

    # ---- does the file even exist? -----------------------------------------
    # Check this first and fail with a clear message. Otherwise pandas throws a
    # confusing error thirty lines later.
    if not DATA.exists():
        # raise SystemExit stops the script immediately with a message.
        raise SystemExit(f"Cannot find the data file at:\n  {DATA}\n"
                         f"It is gitignored (92 MB), so on a fresh clone you "
                         f"need to download it from the PhoNIX release first.")

    # ---- load it ------------------------------------------------------------
    # A "DataFrame" is pandas' name for a table: rows and named columns, like a
    # spreadsheet. `df` is the near-universal variable name for one.
    #
    # usecols=[...] loads ONLY these columns. The full file has 31 columns and
    # some of them hold entire phonon spectra as long text strings - loading
    # those would use a lot of memory for no benefit here. Ask only for what
    # you need.
    #
    # low_memory=False tells pandas to read the whole column before deciding
    # what type it is. Without it, pandas guesses from the first chunk and can
    # decide a numeric column is text because row 5000 looked odd.
    print(f"\nReading {DATA.name} ...")
    df = pd.read_csv(
        DATA,
        usecols=["mp_id", "formula", "spg_number", "natoms_prim", "volume",
                 "kp", "kc", "klat", "max_phfreq", "fc2_error", "fc3_error"],
        low_memory=False,
    )

    # .shape gives (number of rows, number of columns) as a pair.
    print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns\n")
    print("-" * 78)
    print("CHECK 1: does the file match the paper?")
    print("-" * 78)

    # ---- check 1: size and duplicates ---------------------------------------
    # len(df) counts rows.
    record("rows in the file", len(df), PAPER_N_POINTS,
           "one fewer than the paper is expected - see note below")

    # df["mp_id"] pulls out one column. .nunique() counts distinct values in it.
    n_materials = df["mp_id"].nunique()
    record("unique materials (mp_id)", n_materials, PAPER_N_MATERIALS)

    # Some materials were calculated more than once (different settings). Those
    # repeats are why rows > materials.
    n_repeats = len(df) - n_materials
    record("repeated rows", n_repeats, PAPER_N_DUPLICATES - 1,
           "these must never be split across train and test")

    # ---- check 2: THE MOST IMPORTANT ONE ------------------------------------
    print("\n" + "-" * 78)
    print("CHECK 2: do the two heat channels actually add up?")
    print("-" * 78)
    print("  This decides whether we can build kp + kc = klat into the model.")

    # Arithmetic on a whole column at once. numpy/pandas apply it to every row
    # - no loop needed. This is the single biggest difference from ordinary
    # Python, and it is why these libraries exist.
    #
    # We compare RELATIVE error (divide by klat), not absolute. klat spans from
    # 0.000003 to 27,000 W/mK, so an absolute error of 0.01 would be negligible
    # at the top of that range and catastrophic at the bottom.
    additivity_error = np.abs(df["kp"] + df["kc"] - df["klat"]) / df["klat"]
    worst = float(np.nanmax(additivity_error))   # nanmax ignores missing values

    # 1e-10 is scientific notation for 0.0000000001. Anything smaller than that
    # is just the rounding error of computer arithmetic, not a real discrepancy.
    record("worst relative deviation kp+kc vs klat", f"{worst:.2e}",
           note="below 1e-10 means exactly additive")
    if worst < 1e-10:
        print("  -> EXACT. The constrained model design is sound.")
    else:
        print("  -> NOT exact. The constraint would fight the data. Investigate.")

    # ---- check 3: the target we will predict --------------------------------
    print("\n" + "-" * 78)
    print("CHECK 3: what does the thing we want to predict look like?")
    print("-" * 78)

    # Create a NEW column by assigning to a name that does not exist yet.
    df["f_c"] = df["kc"] / (df["kp"] + df["kc"])

    # A COMPARISON on a column gives a column of True/False, one per row.
    # In Python True counts as 1 and False as 0, so .sum() on it counts the
    # Trues and .mean() gives the fraction - a very common trick.
    is_dominant = df["f_c"] >= DOMINANCE_THRESHOLD
    record(f"rows with f_c >= {DOMINANCE_THRESHOLD}", int(is_dominant.sum()), 616,
           f"{100 * is_dominant.mean():.2f}% of rows - a RARE class")

    # 11 materials have exactly zero coherence conductivity. This matters
    # practically: log(0) is undefined, so any log transform of kc must add a
    # small constant first, or those rows become infinity and poison training.
    n_zero_kc = int((df["kc"] == 0).sum())
    record("rows with kc exactly zero", n_zero_kc, 11,
           "log(0) is undefined - never log-transform kc directly")

    # .min() and .max() on a column. The :.2e formats as scientific notation.
    record("kp range (W/mK)",
           f"{df['kp'].min():.2e} to {df['kp'].max():.2e}",
           note="ten orders of magnitude - work in log space")

    # ---- check 4: collapse repeats to one row per material -------------------
    print("\n" + "-" * 78)
    print("CHECK 4: one row per material")
    print("-" * 78)
    print("  A material calculated twice must not land in both train and test,")
    print("  so we collapse repeats now, using the median of each repeat group.")

    # .groupby("mp_id") gathers rows that share an mp_id. .agg({...}) then says
    # what to do with each column within each group. "median" = middle value,
    # which is more robust to one odd repeat than the average would be.
    # "first" just takes whichever came first, fine for labels like the formula.
    per_material = df.groupby("mp_id").agg({
        "formula": "first",
        "spg_number": "first",
        "natoms_prim": "median",
        "volume": "median",
        "kp": "median",
        "kc": "median",
        "klat": "median",
        "f_c": "median",
    }).reset_index()   # turns mp_id back from an index into a normal column

    dom = per_material["f_c"] >= DOMINANCE_THRESHOLD
    record("materials after collapsing repeats", len(per_material), PAPER_N_MATERIALS)
    record("coherence-dominated materials", int(dom.sum()), 544,
           f"{100 * dom.mean():.1f}% - this is our positive class")

    # Does a big cell predict dominance on its own? If the answer were "yes,
    # completely", the whole project would collapse into "count the atoms" and
    # there would be no paper. We check the crudest version of that here.
    #
    # per_material[dom] keeps only the rows where dom is True - "boolean
    # indexing", one of the most useful things pandas does.
    record("median atoms, all materials", int(per_material["natoms_prim"].median()), 16)
    record("median atoms, dominant only", int(per_material[dom]["natoms_prim"].median()), 32,
           "bigger, but 2x is not a complete explanation")
    record("median klat, dominant only (W/mK)",
           round(float(per_material[dom]["klat"].median()), 3), 0.287,
           "dominant materials really are poor conductors")

    # ---- check 5: how the two conditions relate -----------------------------
    print("\n" + "-" * 78)
    print("CHECK 5: is 'coherence-dominated' just a synonym for 'low kappa'?")
    print("-" * 78)

    low_kappa = per_material["klat"] < 1.0

    # The & symbol means "and", applied row by row. (In pandas you must use
    # & rather than the English word `and`, and each side needs its own
    # brackets - a classic beginner trap.)
    both = int((low_kappa & dom).sum())
    print(f"  materials with klat < 1 W/mK  : {int(low_kappa.sum()):>5}")
    print(f"  materials with f_c >= 0.5     : {int(dom.sum()):>5}")
    print(f"  both                          : {both:>5}")
    print()
    print(f"  P(low kappa | dominant) = {both / int(dom.sum()):.3f}"
          f"   <- dominance almost guarantees low kappa")
    print(f"  P(dominant | low kappa) = {both / int(low_kappa.sum()):.3f}"
          f"   <- but low kappa does NOT imply dominance")
    print()
    print(f"  So f_c picks out a {both}-material subclass from the "
          f"{int(low_kappa.sum())} low-kappa ones.")
    print(f"  That asymmetry is the real research question: not 'which materials")
    print(f"  conduct heat badly', but 'which of the bad conductors do it by the")
    print(f"  wave-like channel'.")

    record("P(low kappa | dominant)", round(both / int(dom.sum()), 3))
    record("P(dominant | low kappa)", round(both / int(low_kappa.sum()), 3))

    # ---- save the results ---------------------------------------------------
    # .mkdir() creates the folder. exist_ok=True means "do not complain if it is
    # already there", which lets the script be re-run safely.
    RESULTS.mkdir(exist_ok=True)

    # pd.DataFrame(list_of_dicts) turns our collected checks into a table.
    # index=False stops pandas writing an extra unnamed row-number column.
    audit = pd.DataFrame(checks)
    audit.to_csv(RESULTS / "01_audit_summary.csv", index=False)

    # ---- draw the picture ---------------------------------------------------
    make_figure(per_material, dom)

    print("\n" + "=" * 78)
    n_bad = int((audit["status"] == "MISMATCH").sum())
    if n_bad == 0:
        print("VERDICT: every check matches the paper. The file is trustworthy.")
    else:
        print(f"VERDICT: {n_bad} check(s) did not match - read the table before continuing.")
    print("=" * 78)
    print(f"\nwrote {RESULTS / '01_audit_summary.csv'}")
    print(f"wrote {RESULTS / '01_target_distribution.png'}")


def make_figure(per_material, dom):
    """Two panels showing what we are up against.

    Left : how rare the positive class is, and where the threshold cuts.
    Right: dominance against cell size, the obvious competing explanation.
    """
    # plt.subplots(1, 2) makes one row of two plots. It hands back the whole
    # figure and the individual axes. "ax" is the standard name for one plot's
    # drawing surface.
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

    # Colours defined once, by name, so both panels stay consistent.
    C_POS, C_NEG, INK, GRID = "#1baf7a", "#8a8a85", "#0b0b0b", "#d8d7d2"

    # --- left panel: the distribution of f_c ---
    ax1.hist(per_material["f_c"], bins=60, color=C_NEG, edgecolor="white", linewidth=0.4)
    ax1.axvline(DOMINANCE_THRESHOLD, color=C_POS, linewidth=2)
    ax1.text(DOMINANCE_THRESHOLD + 0.02, ax1.get_ylim()[1] * 0.75,
             f"threshold\n{int(dom.sum())} materials\nto the right",
             color=C_POS, fontsize=9)
    ax1.set_xlabel("coherence fraction  f_c = kc / (kp + kc)")
    ax1.set_ylabel("number of materials")
    ax1.set_title("Most heat is particle-like; a minority is not", fontsize=11)

    # --- right panel: is it just cell size? ---
    # Two scatter calls, one per class, so they get different colours and the
    # rare class is drawn last (on top) instead of being buried.
    ax2.scatter(per_material[~dom]["natoms_prim"], per_material[~dom]["klat"],
                s=8, color=C_NEG, alpha=0.25, label="particle-dominated")
    ax2.scatter(per_material[dom]["natoms_prim"], per_material[dom]["klat"],
                s=14, color=C_POS, alpha=0.8, label="coherence-dominated")
    ax2.set_xscale("log")   # both axes span orders of magnitude, so use log
    ax2.set_yscale("log")
    ax2.set_xlabel("atoms in the primitive cell")
    ax2.set_ylabel("total conductivity klat (W/mK)")
    ax2.set_title("Big cells help, but do not decide it", fontsize=11)
    ax2.legend(frameon=False, fontsize=9, loc="lower left")

    # Shared styling. A `for` loop applies the same commands to both panels
    # rather than writing them out twice.
    for ax in (ax1, ax2):
        ax.grid(True, color=GRID, linewidth=0.6, alpha=0.7)
        ax.set_axisbelow(True)                      # grid behind the data
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)      # drop the top/right border
        ax.tick_params(labelsize=9)

    fig.tight_layout()                              # stop labels overlapping
    fig.savefig(RESULTS / "01_target_distribution.png", dpi=170, facecolor="white")


# ============================================================================
# THE STARTING LINE
# ============================================================================
# This looks cryptic but the idea is simple. Python sets __name__ to
# "__main__" when you RUN a file directly, and to the module's name when
# another file IMPORTS it. So this line means:
#
#     "only actually do the work if someone ran this file on purpose"
#
# Without it, importing anything from this script would immediately run the
# whole audit as a side effect.
if __name__ == "__main__":
    main()
